In [ ]:
# 从纯文本导入 英文 分节经文 到数据库

import sqlite3
import os
import re

conn = sqlite3.connect("db/bible.db")
cursor = conn.cursor()

def get_book_id(abbr_en):
    cur = cursor.execute(
        "SELECT id FROM book WHERE abbr_en = ?",
        (abbr_en,)
    )
    row = cur.fetchone()
    if not row:
        raise ValueError(f"❌ 数据库中未找到书卷缩写：{abbr_en}")
    return row[0]

def parse_filename(filename):
    name = os.path.splitext(filename)[0].replace("_en", "")
    match = re.match(r"([A-Za-z0-9]+)\_(\d+)", name)
    if not match:
        raise ValueError(f"文件名格式错误：{filename}")

    book_abbr = match.group(1)
    chapter = int(match.group(2))
    book_id = get_book_id(book_abbr)
    return book_abbr, chapter, book_id


def import_verses_from_file(filepath):
    book_abbr, chapter, book_id = parse_filename(os.path.basename(filepath))

    with open(filepath, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    for i, text_en in enumerate(lines, start=1):
        verse_id = f"{book_abbr}.{chapter}.{i}"

        cur = cursor.execute(
            "SELECT text_en FROM verse WHERE id = ?", (verse_id,)
        )
        row = cur.fetchone()

        if row is None:
            cursor.execute("""
                INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (verse_id, book_id, chapter, i, text_en, ""))
            print(f"➕ 新增 {verse_id}")

        elif row[0] == text_en:
            pass

        else:
            ans = input(f"\n⚠️  已存在 {verse_id}，是否覆盖？(Y/N): ").strip().upper()
            if ans == "Y":
                cursor.execute(
                    "UPDATE verse SET text_en = ? WHERE id = ?",
                    (text_en, verse_id)
                )
                print(f"♻️  已覆盖 {verse_id}")
            else:
                print(f"⏭️  跳过 {verse_id}")

    conn.commit()
    print(f"\n✅ {book_abbr}.{chapter} 导入完成")


# 只问文件名
if __name__ == "__main__":
    filename = input("请输入文件名（如 Mt.1.txt）：").strip()
    filepath = os.path.join("outputs", "plaintext", filename)

    if not os.path.exists(filepath):
        print(f"❌ 文件不存在：{filepath}")
    else:
        import_verses_from_file(filepath)

In [11]:
# import_cn.py
# 从纯文本导入中文分节经文（text_cn）
# 文件名规范：2K_4_cn.txt

import sqlite3
import os
import re

# ==========================
# 数据库连接
# ==========================
conn = sqlite3.connect("db/bible.db")
cursor = conn.cursor()

# ==========================
# 从 book 表获取 book_id
# ==========================
def get_book_id(abbr_en):
    cur = cursor.execute(
        "SELECT id FROM book WHERE abbr_en = ?",
        (abbr_en,)
    )
    row = cur.fetchone()
    if not row:
        raise ValueError(f"❌ 数据库中未找到书卷缩写：{abbr_en}")
    return row[0]

# ==========================
# 解析文件名（2K_4_cn.txt）
# ==========================
def parse_filename(filename):
    name = os.path.splitext(filename)[0]          # 去掉 .txt
    name = name.replace("_cn", "")                 # 去掉 _cn

    match = re.match(r"([A-Za-z0-9]+)\_(\d+)", name)
    if not match:
        raise ValueError(f"文件名格式错误：{filename}")

    book_abbr = match.group(1)   # 2K
    chapter = int(match.group(2))  # 4
    book_id = get_book_id(book_abbr)

    return book_abbr, chapter, book_id

# ==========================
# 导入中文经文
# ==========================
def import_verses_from_file(filepath):
    book_abbr, chapter, book_id = parse_filename(os.path.basename(filepath))

    with open(filepath, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    for i, text_cn in enumerate(lines, start=1):
        verse_id = f"{book_abbr}.{chapter}.{i}"

        cur = cursor.execute(
            "SELECT text_cn FROM verse WHERE id = ?",
            (verse_id,)
        )
        row = cur.fetchone()

        # verse 不存在（极少见）
        if row is None:
            cursor.execute("""
                INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (verse_id, book_id, chapter, i, "", text_cn))
            print(f"➕ 新增 {verse_id}")

        elif row[0] is None or row[0] == "":
            cursor.execute(
                "UPDATE verse SET text_cn = ? WHERE id = ?",
                (text_cn, verse_id)
            )
            print(f"➕ 写入 text_cn: {verse_id}")

        else:
            ans = input(f"\n⚠️  已存在 {verse_id}，是否覆盖中文译文？(Y/N): ").strip().upper()
            if ans == "Y":
                cursor.execute(
                    "UPDATE verse SET text_cn = ? WHERE id = ?",
                    (text_cn, verse_id)
                )
                print(f"♻️  已覆盖 text_cn: {verse_id}")
            else:
                print(f"⏭️  跳过 {verse_id}")

    conn.commit()
    print(f"\n✅ {book_abbr}.{chapter} 中文经文导入完成")

# ==========================
# 主程序
# ==========================
if __name__ == "__main__":
    filename = input("请输入中文经文文件名（如 2K_4_cn.txt）：").strip()
    filepath = os.path.join("outputs", "plaintext", filename)

    if not os.path.exists(filepath):
        print(f"❌ 文件不存在：{filepath}")
    else:
        import_verses_from_file(filepath)

请输入中文经文文件名（如 2K_4_cn.txt）：Mt_10_cn.txt
➕ 写入 text_cn: Mt.10.1
➕ 写入 text_cn: Mt.10.2
➕ 写入 text_cn: Mt.10.3
➕ 写入 text_cn: Mt.10.4
➕ 写入 text_cn: Mt.10.5
➕ 写入 text_cn: Mt.10.6
➕ 写入 text_cn: Mt.10.7
➕ 写入 text_cn: Mt.10.8
➕ 写入 text_cn: Mt.10.9
➕ 写入 text_cn: Mt.10.10
➕ 写入 text_cn: Mt.10.11
➕ 写入 text_cn: Mt.10.12
➕ 写入 text_cn: Mt.10.13
➕ 写入 text_cn: Mt.10.14
➕ 写入 text_cn: Mt.10.15
➕ 写入 text_cn: Mt.10.16
➕ 写入 text_cn: Mt.10.17
➕ 写入 text_cn: Mt.10.18
➕ 写入 text_cn: Mt.10.19
➕ 写入 text_cn: Mt.10.20
➕ 写入 text_cn: Mt.10.21
➕ 写入 text_cn: Mt.10.22
➕ 写入 text_cn: Mt.10.23
➕ 写入 text_cn: Mt.10.24
➕ 写入 text_cn: Mt.10.25
➕ 写入 text_cn: Mt.10.26
➕ 写入 text_cn: Mt.10.27
➕ 写入 text_cn: Mt.10.28
➕ 写入 text_cn: Mt.10.29
➕ 写入 text_cn: Mt.10.30
➕ 写入 text_cn: Mt.10.31
➕ 写入 text_cn: Mt.10.32
➕ 写入 text_cn: Mt.10.33
➕ 写入 text_cn: Mt.10.34
➕ 写入 text_cn: Mt.10.35
➕ 写入 text_cn: Mt.10.36
➕ 写入 text_cn: Mt.10.37
➕ 写入 text_cn: Mt.10.38
➕ 写入 text_cn: Mt.10.39
➕ 写入 text_cn: Mt.10.40
➕ 写入 text_cn: Mt.10.41
➕ 写入 text_cn: Mt.10.42

✅ M

In [ ]:
# 从纯文本导入英文分节经文到数据库（按书卷缩写）

import sqlite3
import os
import re

conn = sqlite3.connect("db/bible.db")
cursor = conn.cursor()

# ========== 配置 ==========
PLAINTEXT_DIR = "outputs/plaintext"

# ========== 工具函数 ==========
def get_book_info(abbr_en):
    cursor.execute(
        "SELECT id, name_en, max_chapter FROM book WHERE abbr_en = ?",
        (abbr_en,)
    )
    row = cursor.fetchone()
    if not row:
        raise ValueError(f"❌ 数据库中未找到书卷缩写：{abbr_en}")
    return row  # (id, name_en, max_chapter)


def find_chapter_files(abbr_en, max_chapter):
    """
    在 outputs/plaintext 中查找：
    abbr_en_1_en.txt ~ abbr_en_{max_chapter}_en.txt
    """
    files = []
    for ch in range(1, max_chapter + 1):
        filename = f"{abbr_en}_{ch}_en.txt"
        filepath = os.path.join(PLAINTEXT_DIR, filename)
        if os.path.exists(filepath):
            files.append((ch, filepath))
        else:
            print(f"⚠️ 缺少文件：{filename}")
    return files


def import_chapter(book_id, chapter, filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    inserted = 0

    for i, text_en in enumerate(lines, start=1):
        verse_id = f"{book_id}.{chapter}.{i}"

        cursor.execute(
            "SELECT 1 FROM verse WHERE id = ?",
            (verse_id,)
        )

        if cursor.fetchone() is None:
            cursor.execute("""
                INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (verse_id, book_id, chapter, i, text_en, ""))
            inserted += 1

    conn.commit()
    return inserted


# ========== 主流程 ==========
if __name__ == "__main__":
    abbr_en = input("请输入书卷缩写（如 Gen、Matt）：").strip()

    try:
        book_id, name_en, max_chapter = get_book_info(abbr_en)
        print(f"\n📖 书卷：{name_en}（{abbr_en}），共 {max_chapter} 章")

        chapters = find_chapter_files(abbr_en, max_chapter)

        if not chapters:
            print("❌ 没有找到任何可导入的章节文件")
        else:
            total_inserted = 0
            for chapter, filepath in chapters:
                count = import_chapter(book_id, chapter, filepath)
                print(f"  ✅ {abbr_en} {chapter}：新增 {count} 节")
                total_inserted += count

            print(f"\n🎉 导入完成！共新增 {total_inserted} 节经文")

    except Exception as e:
        print(e)

In [ ]:
import sqlite3
import os

# ========== 配置 ==========
DB_PATH = "db/bible.db"
PLAINTEXT_DIR = "outputs/plaintext"

# ========== 数据库连接 ==========
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# ---------- 确保进度表 ----------
cursor.execute("""
CREATE TABLE IF NOT EXISTS import_progress (
    abbr_en TEXT,
    chapter INTEGER,
    done INTEGER DEFAULT 0,
    PRIMARY KEY (abbr_en, chapter)
)
""")
conn.commit()

# ========== 工具函数 ==========
def get_book_info(abbr_en):
    cursor.execute(
        "SELECT id, name_en, max_chapter FROM book WHERE abbr_en = ?",
        (abbr_en,)
    )
    row = cursor.fetchone()
    if not row:
        raise ValueError(f"❌ 未找到书卷：{abbr_en}")
    return row


def is_chapter_imported(abbr_en, chapter):
    cursor.execute(
        "SELECT 1 FROM import_progress WHERE abbr_en = ? AND chapter = ? AND done = 1",
        (abbr_en, chapter)
    )
    return cursor.fetchone() is not None


def mark_chapter_done(abbr_en, chapter):
    cursor.execute("""
        INSERT INTO import_progress (abbr_en, chapter, done)
        VALUES (?, ?, 1)
        ON CONFLICT(abbr_en, chapter) DO UPDATE SET done = 1
    """, (abbr_en, chapter))
    conn.commit()


def find_chapter_files(abbr_en, max_chapter):
    files = []
    for ch in range(1, max_chapter + 1):
        filename = f"{abbr_en}_{ch}_en.txt"
        filepath = os.path.join(PLAINTEXT_DIR, filename)
        if os.path.exists(filepath):
            files.append((ch, filepath))
        else:
            print(f"⚠️ 缺少文件：{filename}")
    return files


def import_chapter(book_id, abbr_en, chapter, filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    inserted = 0
    for i, text_en in enumerate(lines, start=1):
        verse_id = f"{abbr_en}.{chapter}.{i}"
        cursor.execute("SELECT 1 FROM verse WHERE id = ?", (verse_id,))
        if cursor.fetchone():
            continue

        cursor.execute("""
            INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (verse_id, book_id, chapter, i, text_en, ""))
        inserted += 1

    conn.commit()
    mark_chapter_done(abbr_en, chapter)
    return inserted

# ========== 主流程 ==========
if __name__ == "__main__":
    abbr_en = input("请输入书卷缩写（如 Gen、Matt）：").strip()

    try:
        book_id, name_en, max_chapter = get_book_info(abbr_en)
        print(f"\n📖 开始导入书卷：{name_en}（{abbr_en}），共 {max_chapter} 章")

        chapters = find_chapter_files(abbr_en, max_chapter)
        total_inserted = 0
        skipped = 0

        for idx, (chapter, filepath) in enumerate(chapters, start=1):
            if is_chapter_imported(abbr_en, chapter):
                print(f"[{idx}/{len(chapters)}] ⏭️ 跳过 {abbr_en} {chapter}")
                skipped += 1
                continue

            print(f"[{idx}/{len(chapters)}] 📥 导入 {abbr_en} {chapter} ...", end=" ")
            count = import_chapter(book_id, abbr_en, chapter, filepath)
            total_inserted += count
            print(f"✅ +{count} 节")

        print("\n🎉 导入完成！")
        print(f"   ➕ 新增经文：{total_inserted} 节")
        print(f"   ⏭️ 跳过已导入：{skipped} 章")

    except Exception as e:
        print(f"\n❌ 错误：{e}")

In [1]:
# 从 outputs/plaintext 遍历所有纯文本导入英文，增量更新

import sqlite3
import os
import re

# ========== 配置 ==========
DB_PATH = "db/bible.db"
PLAINTEXT_DIR = "mismatch/plaintext"

# ========== 数据库初始化 ==========
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS import_progress (
    abbr_en TEXT,
    chapter INTEGER,
    done INTEGER DEFAULT 0,
    PRIMARY KEY (abbr_en, chapter)
)
""")
conn.commit()

# ========== 工具函数 ==========
def get_book_id(abbr_en):
    cursor.execute(
        "SELECT id FROM book WHERE abbr_en = ?",
        (abbr_en,)
    )
    row = cursor.fetchone()
    if not row:
        raise ValueError(f"❌ 数据库中未找到书卷缩写：{abbr_en}")
    return row[0]


def is_chapter_imported(abbr_en, chapter):
    cursor.execute(
        "SELECT 1 FROM import_progress WHERE abbr_en = ? AND chapter = ? AND done = 1",
        (abbr_en, chapter)
    )
    return cursor.fetchone() is not None


def mark_chapter_done(abbr_en, chapter):
    cursor.execute("""
        INSERT INTO import_progress (abbr_en, chapter, done)
        VALUES (?, ?, 1)
        ON CONFLICT(abbr_en, chapter) DO UPDATE SET done = 1
    """, (abbr_en, chapter))
    conn.commit()


def import_chapter(abbr_en, chapter, filepath):
    book_id = get_book_id(abbr_en)

    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    inserted = 0

    for i, text_en in enumerate(lines, start=1):
        verse_id = f"{abbr_en}.{chapter}.{i}"

        cursor.execute(
            "SELECT 1 FROM verse WHERE id = ?",
            (verse_id,)
        )
        if cursor.fetchone():
            continue

        cursor.execute("""
            INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (verse_id, book_id, chapter, i, text_en, ""))
        inserted += 1

    conn.commit()
    mark_chapter_done(abbr_en, chapter)
    return inserted


# ========== 主流程 ==========
if __name__ == "__main__":
    # 匹配：{book_abbr}_{chapter}_en.txt
    pattern = re.compile(r"^([A-Za-z0-9]+)_(\d+)_en\.txt$")

    files = []
    for filename in os.listdir(PLAINTEXT_DIR):
        match = pattern.match(filename)
        if not match:
            continue
        abbr_en = match.group(1)
        chapter = int(match.group(2))
        files.append((abbr_en, chapter, os.path.join(PLAINTEXT_DIR, filename)))

    if not files:
        print("❌ 未找到任何匹配的 txt 文件")
    else:
        total_inserted = 0
        skipped = 0

        for idx, (abbr_en, chapter, filepath) in enumerate(files, start=1):
            if is_chapter_imported(abbr_en, chapter):
                print(f"[{idx}/{len(files)}] ⏭️ 跳过 {abbr_en} {chapter}")
                skipped += 1
                continue

            print(f"[{idx}/{len(files)}] 📥 导入 {abbr_en} {chapter} ...", end=" ")
            count = import_chapter(abbr_en, chapter, filepath)
            total_inserted += count
            print(f"✅ +{count} 节")

        print("\n🎉 全部导入完成！")
        print(f"   ➕ 新增经文：{total_inserted} 节")
        print(f"   ⏭️ 跳过已导入：{skipped} 章")

[1/230] 📥 导入 Es 6 ... ✅ +14 节
[2/230] 📥 导入 Acts 15 ... ✅ +41 节
[3/230] 📥 导入 Sir 31 ... ✅ +31 节
[4/230] 📥 导入 Sir 40 ... ✅ +30 节
[5/230] 📥 导入 Lev 5 ... ✅ +19 节
[6/230] 📥 导入 Acts 19 ... ✅ +41 节
[7/230] 📥 导入 Mk 9 ... ✅ +49 节
[8/230] 📥 导入 Sir 23 ... ✅ +27 节
[9/230] 📥 导入 Jer 9 ... ✅ +26 节
[10/230] 📥 导入 Sir 15 ... ✅ +20 节
[11/230] 📥 导入 Sir 19 ... ✅ +30 节
[12/230] 📥 导入 Ne 9 ... ✅ +38 节
[13/230] 📥 导入 1K 5 ... ✅ +18 节
[14/230] 📥 导入 Nh 1 ... ✅ +15 节
[15/230] 📥 导入 Mk 15 ... ✅ +47 节
[16/230] 📥 导入 Ps 26 ... ✅ +12 节
[17/230] 📥 导入 Wis 6 ... ✅ +25 节
[18/230] 📥 导入 Ps 130 ... ✅ +8 节
[19/230] 📥 导入 Ps 141 ... ✅ +10 节
[20/230] 📥 导入 Ps 73 ... ✅ +28 节
[21/230] 📥 导入 Sir 7 ... ✅ +36 节
[22/230] 📥 导入 Job 27 ... ✅ +23 节
[23/230] 📥 导入 Ps 122 ... ✅ +9 节
[24/230] 📥 导入 Dt 12 ... ✅ +32 节
[25/230] 📥 导入 Ps 28 ... ✅ +9 节
[26/230] 📥 导入 Bar 6 ... ✅ +73 节
[27/230] 📥 导入 Ps 24 ... ✅ +10 节
[28/230] 📥 导入 Tb 10 ... ✅ +13 节
[29/230] 📥 导入 Ex 8 ... ✅ +32 节
[30/230] 📥 导入 Ps 120 ... ✅ +7 节
[31/230] 📥 导入 Dt 28 ... ✅ +68 节
[32/230] 📥 导入

In [16]:
"""
import_cn.py
从纯文本导入中文分节经文（text_cn）
文件名规范：Gen_1_cn.txt
支持增量更新、断点续传
"""

import sqlite3
import os
import re

# ==========================
# 配置
# ==========================
DB_PATH = "db/bible.db"
PLAINTEXT_DIR = "outputs/plaintext_cn"

# ==========================
# 数据库连接
# ==========================
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# ==========================
# 导入进度表（中文专用）
# ==========================
cursor.execute("""
CREATE TABLE IF NOT EXISTS import_progress_cn (
    abbr_en TEXT,
    chapter INTEGER,
    done INTEGER DEFAULT 0,
    PRIMARY KEY (abbr_en, chapter)
)
""")
conn.commit()

# ==========================
# 工具函数
# ==========================
def get_book_id(abbr_en):
    cursor.execute(
        "SELECT id FROM book WHERE abbr_en = ?",
        (abbr_en,)
    )
    row = cursor.fetchone()
    if not row:
        raise ValueError(f"❌ 数据库中未找到书卷缩写：{abbr_en}")
    return row[0]


def is_chapter_imported(abbr_en, chapter):
    cursor.execute(
        "SELECT 1 FROM import_progress_cn WHERE abbr_en = ? AND chapter = ? AND done = 1",
        (abbr_en, chapter)
    )
    return cursor.fetchone() is not None


def mark_chapter_done(abbr_en, chapter):
    cursor.execute("""
        INSERT INTO import_progress_cn (abbr_en, chapter, done)
        VALUES (?, ?, 1)
        ON CONFLICT(abbr_en, chapter) DO UPDATE SET done = 1
    """, (abbr_en, chapter))
    conn.commit()


def import_chapter(abbr_en, chapter, filepath):
    book_id = get_book_id(abbr_en)

    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    stats = {
        "created": 0,   # verse 原本不存在
        "filled": 0,    # text_cn 为空，被写入
        "skipped": 0,   # text_cn 已存在
    }

    for i, text_cn in enumerate(lines, start=1):
        verse_id = f"{abbr_en}.{chapter}.{i}"

        cursor.execute(
            "SELECT text_cn FROM verse WHERE id = ?",
            (verse_id,)
        )
        row = cursor.fetchone()

        # 1️⃣ verse 不存在（中文比英文全）
        if row is None:
            cursor.execute("""
                INSERT INTO verse (
                    id, book_id, chapter, verse,
                    text_en, text_cn
                ) VALUES (?, ?, ?, ?, ?, ?)
            """, (verse_id, book_id, chapter, i, "", text_cn))
            stats["created"] += 1
            continue

        # 2️⃣ text_cn 为空，写入
        if row[0] is None or row[0] == "":
            cursor.execute(
                "UPDATE verse SET text_cn = ? WHERE id = ?",
                (text_cn, verse_id)
            )
            stats["filled"] += 1
            continue

        # 3️⃣ 已有中文译文
        stats["skipped"] += 1

    conn.commit()
    mark_chapter_done(abbr_en, chapter)
    return stats

# ==========================
# 主流程
# ==========================
if __name__ == "__main__":
    # 匹配：{book_abbr}_{chapter}_cn.txt
    pattern = re.compile(r"^([A-Za-z0-9]+)_(\d+)_cn\.txt$")

    files = []
    for filename in os.listdir(PLAINTEXT_DIR):
        match = pattern.match(filename)
        if not match:
            continue
        abbr_en = match.group(1)
        chapter = int(match.group(2))
        files.append((abbr_en, chapter, os.path.join(PLAINTEXT_DIR, filename)))

    if not files:
        print("❌ 未找到任何中文经文文件")
    else:
        total_created = 0
        total_filled = 0
        total_skipped = 0
        skipped_chapters = 0

        for idx, (abbr_en, chapter, filepath) in enumerate(files, start=1):
            if is_chapter_imported(abbr_en, chapter):
                print(f"[{idx}/{len(files)}] ⏭️ 跳过 {abbr_en} {chapter}")
                skipped_chapters += 1
                continue

            print(f"[{idx}/{len(files)}] 📥 导入 {abbr_en} {chapter} ...", end=" ")
            stats = import_chapter(abbr_en, chapter, filepath)
            total_created += stats["created"]
            total_filled += stats["filled"]
            total_skipped += stats["skipped"]

            print(
                f"✅ 新建 {stats['created']} / 写入 {stats['filled']} / 跳过 {stats['skipped']}"
            )

        print("\n🎉 中文经文导入完成！")
        print(f"   ➕ 新建 verse（英文未导入）：{total_created}")
        print(f"   ✍️ 写入 text_cn：{total_filled}")
        print(f"   ⏭️ 跳过已有中文：{total_skipped}")
        print(f"   📦 跳过已完成章节：{skipped_chapters}")

[1/1334] 📥 导入 Ps 86 ... ✅ 新建 17 / 写入 0 / 跳过 0
[2/1334] 📥 导入 Is 36 ... ✅ 新建 0 / 写入 22 / 跳过 0
[3/1334] 📥 导入 Gen 2 ... ✅ 新建 0 / 写入 25 / 跳过 0
[4/1334] 📥 导入 Num 32 ... ✅ 新建 0 / 写入 42 / 跳过 0
[5/1334] 📥 导入 Sir 13 ... ✅ 新建 32 / 写入 0 / 跳过 0
[6/1334] 📥 导入 1Mac 8 ... ✅ 新建 0 / 写入 32 / 跳过 0
[7/1334] 📥 导入 Mt 17 ... ✅ 新建 27 / 写入 0 / 跳过 0
[8/1334] 📥 导入 1S 9 ... ✅ 新建 0 / 写入 27 / 跳过 0
[9/1334] 📥 导入 Is 28 ... ✅ 新建 0 / 写入 29 / 跳过 0
[10/1334] 📥 导入 Ne 3 ... ✅ 新建 38 / 写入 0 / 跳过 0
[11/1334] 📥 导入 Jdg 7 ... ✅ 新建 0 / 写入 25 / 跳过 0
[12/1334] 📥 导入 Ezk 19 ... ✅ 新建 0 / 写入 14 / 跳过 0
[13/1334] 📥 导入 Jdg 19 ... ✅ 新建 0 / 写入 30 / 跳过 0
[14/1334] 📥 导入 Ps 98 ... ✅ 新建 9 / 写入 0 / 跳过 0
[15/1334] 📥 导入 Hg 2 ... ✅ 新建 23 / 写入 0 / 跳过 0
[16/1334] 📥 导入 Is 55 ... ✅ 新建 0 / 写入 13 / 跳过 0
[17/1334] 📥 导入 Rev 3 ... ✅ 新建 0 / 写入 22 / 跳过 0
[18/1334] 📥 导入 Ezk 15 ... ✅ 新建 0 / 写入 8 / 跳过 0
[19/1334] 📥 导入 Jdg 15 ... ✅ 新建 0 / 写入 20 / 跳过 0
[20/1334] 📥 导入 1K 3 ... ✅ 新建 0 / 写入 28 / 跳过 0
[21/1334] ⏭️ 跳过 Mt 1
[22/1334] 📥 导入 Ps 94 ... ✅ 新建 0 / 写入 23 / 跳过 0


[182/1334] 📥 导入 2P 3 ... ✅ 新建 0 / 写入 18 / 跳过 0
[183/1334] 📥 导入 Jos 20 ... ✅ 新建 0 / 写入 9 / 跳过 0
[184/1334] 📥 导入 Ps 18 ... ✅ 新建 0 / 写入 51 / 跳过 0
[185/1334] 📥 导入 Jdt 15 ... ✅ 新建 0 / 写入 14 / 跳过 0
[186/1334] 📥 导入 Ex 35 ... ✅ 新建 0 / 写入 35 / 跳过 0
[187/1334] 📥 导入 1Mac 12 ... ✅ 新建 0 / 写入 53 / 跳过 0
[188/1334] 📥 导入 1Jn 3 ... ✅ 新建 0 / 写入 24 / 跳过 0
[189/1334] 📥 导入 1Chr 23 ... ✅ 新建 0 / 写入 32 / 跳过 0
[190/1334] 📥 导入 Job 31 ... ✅ 新建 0 / 写入 40 / 跳过 0
[191/1334] 📥 导入 2Chr 12 ... ✅ 新建 0 / 写入 16 / 跳过 0
[192/1334] 📥 导入 Ps 65 ... ✅ 新建 0 / 写入 14 / 跳过 0
[193/1334] 📥 导入 Ecl 7 ... ✅ 新建 0 / 写入 29 / 跳过 0
[194/1334] 📥 导入 Ps 134 ... ✅ 新建 3 / 写入 0 / 跳过 0
[195/1334] 📥 导入 Mic 6 ... ✅ 新建 0 / 写入 16 / 跳过 0
[196/1334] 📥 导入 Lk 12 ... ✅ 新建 0 / 写入 59 / 跳过 0
[197/1334] 📥 导入 Dt 22 ... ✅ 新建 29 / 写入 0 / 跳过 0
[198/1334] 📥 导入 Jos 9 ... ✅ 新建 0 / 写入 27 / 跳过 0
[199/1334] 📥 导入 Ps 149 ... ✅ 新建 0 / 写入 9 / 跳过 0
[200/1334] 📥 导入 2Mac 15 ... ✅ 新建 0 / 写入 39 / 跳过 0
[201/1334] 📥 导入 Ezk 9 ... ✅ 新建 0 / 写入 11 / 跳过 0
[202/1334] 📥 导入 Gen 32 ... ✅ 新建

[371/1334] 📥 导入 Ps 34 ... ✅ 新建 0 / 写入 23 / 跳过 0
[372/1334] 📥 导入 Ex 19 ... ✅ 新建 0 / 写入 25 / 跳过 0
[373/1334] 📥 导入 Lk 20 ... ✅ 新建 0 / 写入 47 / 跳过 0
[374/1334] 📥 导入 Job 4 ... ✅ 新建 0 / 写入 21 / 跳过 0
[375/1334] 📥 导入 Gen 28 ... ✅ 新建 0 / 写入 22 / 跳过 0
[376/1334] 📥 导入 Ru 3 ... ✅ 新建 0 / 写入 18 / 跳过 0
[377/1334] 📥 导入 1Cor 5 ... ✅ 新建 0 / 写入 13 / 跳过 0
[378/1334] 📥 导入 Ps 61 ... ✅ 新建 0 / 写入 9 / 跳过 0
[379/1334] 📥 导入 Ecl 3 ... ✅ 新建 0 / 写入 22 / 跳过 0
[380/1334] 📥 导入 Jos 24 ... ✅ 新建 0 / 写入 33 / 跳过 0
[381/1334] 📥 导入 Jdt 11 ... ✅ 新建 0 / 写入 23 / 跳过 0
[382/1334] 📥 导入 Ex 31 ... ✅ 新建 0 / 写入 18 / 跳过 0
[383/1334] 📥 导入 1Mac 16 ... ✅ 新建 0 / 写入 24 / 跳过 0
[384/1334] 📥 导入 1Chr 27 ... ✅ 新建 0 / 写入 34 / 跳过 0
[385/1334] 📥 导入 Job 35 ... ✅ 新建 0 / 写入 16 / 跳过 0
[386/1334] 📥 导入 2Chr 16 ... ✅ 新建 0 / 写入 14 / 跳过 0
[387/1334] 📥 导入 Gal 4 ... ✅ 新建 0 / 写入 31 / 跳过 0
[388/1334] 📥 导入 Dt 26 ... ✅ 新建 0 / 写入 19 / 跳过 0
[389/1334] 📥 导入 Hb 3 ... ✅ 新建 0 / 写入 19 / 跳过 0
[390/1334] 📥 导入 Phil 4 ... ✅ 新建 0 / 写入 23 / 跳过 0
[391/1334] 📥 导入 Ps 130 ... ✅ 新建

[556/1334] 📥 导入 Rev 18 ... ✅ 新建 0 / 写入 24 / 跳过 0
[557/1334] 📥 导入 Mt 13 ... ✅ 新建 0 / 写入 58 / 跳过 0
[558/1334] 📥 导入 Ps 82 ... ✅ 新建 8 / 写入 0 / 跳过 0
[559/1334] 📥 导入 Gen 6 ... ✅ 新建 0 / 写入 22 / 跳过 0
[560/1334] 📥 导入 Is 32 ... ✅ 新建 0 / 写入 20 / 跳过 0
[561/1334] 📥 导入 Num 36 ... ✅ 新建 0 / 写入 13 / 跳过 0
[562/1334] 📥 导入 Sir 17 ... ✅ 新建 31 / 写入 0 / 跳过 0
[563/1334] 📥 导入 Is 51 ... ✅ 新建 0 / 写入 23 / 跳过 0
[564/1334] 📥 导入 Rev 7 ... ✅ 新建 0 / 写入 17 / 跳过 0
[565/1334] 📥 导入 2S 22 ... ✅ 新建 0 / 写入 51 / 跳过 0
[566/1334] 📥 导入 2Cor 10 ... ✅ 新建 0 / 写入 18 / 跳过 0
[567/1334] 📥 导入 Jdg 3 ... ✅ 新建 0 / 写入 31 / 跳过 0
[568/1334] 📥 导入 Ne 7 ... ✅ 新建 72 / 写入 0 / 跳过 0
[569/1334] 📥 导入 Num 28 ... ✅ 新建 0 / 写入 31 / 跳过 0
[570/1334] 📥 导入 Hos 9 ... ✅ 新建 0 / 写入 17 / 跳过 0
[571/1334] 📥 导入 Mt 9 ... ✅ 新建 0 / 写入 38 / 跳过 0
[572/1334] 📥 导入 Dt 12 ... ✅ 新建 31 / 写入 0 / 跳过 0
[573/1334] 📥 导入 Jn 8 ... ✅ 新建 0 / 写入 59 / 跳过 0
[574/1334] 📥 导入 Heb 4 ... ✅ 新建 0 / 写入 16 / 跳过 0
[575/1334] 📥 导入 Wis 4 ... ✅ 新建 0 / 写入 20 / 跳过 0
[576/1334] 📥 导入 Pro 30 ... ✅ 新建 0 / 写入

[751/1334] 📥 导入 Jer 21 ... ✅ 新建 0 / 写入 14 / 跳过 0
[752/1334] 📥 导入 Mk 2 ... ✅ 新建 0 / 写入 28 / 跳过 0
[753/1334] 📥 导入 Es 1 ... ✅ 新建 17 / 写入 0 / 跳过 0
[754/1334] 📥 导入 Jas 1 ... ✅ 新建 27 / 写入 0 / 跳过 0
[755/1334] 📥 导入 Sir 28 ... ✅ 新建 30 / 写入 0 / 跳过 0
[756/1334] 📥 导入 2K 5 ... ✅ 新建 0 / 写入 27 / 跳过 0
[757/1334] 📥 导入 2S 3 ... ✅ 新建 0 / 写入 39 / 跳过 0
[758/1334] 📥 导入 1K 18 ... ✅ 新建 0 / 写入 46 / 跳过 0
[759/1334] 📥 导入 Is 13 ... ✅ 新建 0 / 写入 22 / 跳过 0
[760/1334] 📥 导入 Wis 15 ... ✅ 新建 0 / 写入 19 / 跳过 0
[761/1334] 📥 导入 Jer 42 ... ✅ 新建 0 / 写入 22 / 跳过 0
[762/1334] 📥 导入 Acts 12 ... ✅ 新建 0 / 写入 25 / 跳过 0
[763/1334] 📥 导入 Num 17 ... ✅ 新建 28 / 写入 0 / 跳过 0
[764/1334] 📥 导入 Zec 12 ... ✅ 新建 0 / 写入 14 / 跳过 0
[765/1334] 📥 导入 Sir 36 ... ✅ 新建 28 / 写入 0 / 跳过 0
[766/1334] 📥 导入 Pro 3 ... ✅ 新建 0 / 写入 35 / 跳过 0
[767/1334] 📥 导入 Ezk 22 ... ✅ 新建 0 / 写入 31 / 跳过 0
[768/1334] 📥 导入 2S 11 ... ✅ 新建 0 / 写入 27 / 跳过 0
[769/1334] 📥 导入 Sir 47 ... ✅ 新建 31 / 写入 0 / 跳过 0
[770/1334] 📥 导入 2Chr 2 ... ✅ 新建 17 / 写入 0 / 跳过 0
[771/1334] 📥 导入 Is 62 ... ✅ 新建 0

[929/1334] 📥 导入 2K 25 ... ✅ 新建 0 / 写入 30 / 跳过 0
[930/1334] 📥 导入 Jn 11 ... ✅ 新建 0 / 写入 57 / 跳过 0
[931/1334] 📥 导入 Hos 10 ... ✅ 新建 0 / 写入 15 / 跳过 0
[932/1334] 📥 导入 Ps 23 ... ✅ 新建 6 / 写入 0 / 跳过 0
[933/1334] 📥 导入 Ezk 4 ... ✅ 新建 0 / 写入 17 / 跳过 0
[934/1334] 📥 导入 2Chr 29 ... ✅ 新建 0 / 写入 36 / 跳过 0
[935/1334] 📥 导入 1P 5 ... ✅ 新建 0 / 写入 14 / 跳过 0
[936/1334] 📥 导入 Lev 10 ... ✅ 新建 0 / 写入 20 / 跳过 0
[937/1334] 📥 导入 1Chr 18 ... ✅ 新建 0 / 写入 17 / 跳过 0
[938/1334] 📥 导入 Dt 1 ... ✅ 新建 0 / 写入 46 / 跳过 0
[939/1334] 📥 导入 Dt 19 ... ✅ 新建 0 / 写入 21 / 跳过 0
[940/1334] 📥 导入 Lm 2 ... ✅ 新建 0 / 写入 22 / 跳过 0
[941/1334] 📥 导入 Jn 3 ... ✅ 新建 0 / 写入 36 / 跳过 0
[942/1334] 📥 导入 Num 6 ... ✅ 新建 0 / 写入 27 / 跳过 0
[943/1334] 📥 导入 Ps 40 ... ✅ 新建 0 / 写入 18 / 跳过 0
[944/1334] 📥 导入 Gen 21 ... ✅ 新建 0 / 写入 34 / 跳过 0
[945/1334] 📥 导入 Job 14 ... ✅ 新建 0 / 写入 22 / 跳过 0
[946/1334] 📥 导入 1Thes 5 ... ✅ 新建 0 / 写入 28 / 跳过 0
[947/1334] 📥 导入 Ex 10 ... ✅ 新建 0 / 写入 29 / 跳过 0
[948/1334] 📥 导入 Jdt 2 ... ✅ 新建 0 / 写入 28 / 跳过 0
[949/1334] 📥 导入 Job 41 ... ✅ 新建 26 

[1126/1334] 📥 导入 1Cor 8 ... ✅ 新建 0 / 写入 13 / 跳过 0
[1127/1334] 📥 导入 Heb 13 ... ✅ 新建 0 / 写入 25 / 跳过 0
[1128/1334] 📥 导入 1Thes 1 ... ✅ 新建 0 / 写入 10 / 跳过 0
[1129/1334] 📥 导入 Ps 39 ... ✅ 新建 0 / 写入 14 / 跳过 0
[1130/1334] 📥 导入 Tb 13 ... ✅ 新建 18 / 写入 0 / 跳过 0
[1131/1334] 📥 导入 Ex 14 ... ✅ 新建 0 / 写入 31 / 跳过 0
[1132/1334] 📥 导入 Dt 5 ... ✅ 新建 0 / 写入 33 / 跳过 0
[1133/1334] 📥 导入 Ecl 12 ... ✅ 新建 0 / 写入 14 / 跳过 0
[1134/1334] 📥 导入 Jn 7 ... ✅ 新建 0 / 写入 53 / 跳过 0
[1135/1334] 📥 导入 Num 2 ... ✅ 新建 0 / 写入 34 / 跳过 0
[1136/1334] 📥 导入 Ps 44 ... ✅ 新建 0 / 写入 27 / 跳过 0
[1137/1334] 📥 导入 Ex 7 ... ✅ 新建 29 / 写入 0 / 跳过 0
[1138/1334] 📥 导入 Ps 35 ... ✅ 新建 28 / 写入 0 / 跳过 0
[1139/1334] 📥 导入 Ex 18 ... ✅ 新建 0 / 写入 27 / 跳过 0
[1140/1334] 📥 导入 Lk 21 ... ✅ 新建 0 / 写入 38 / 跳过 0
[1141/1334] 📥 导入 Ps 107 ... ✅ 新建 0 / 写入 43 / 跳过 0
[1142/1334] 📥 导入 Ru 2 ... ✅ 新建 0 / 写入 23 / 跳过 0
[1143/1334] 📥 导入 1Cor 4 ... ✅ 新建 0 / 写入 21 / 跳过 0
[1144/1334] 📥 导入 Job 5 ... ✅ 新建 0 / 写入 27 / 跳过 0
[1145/1334] 📥 导入 Gen 29 ... ✅ 新建 0 / 写入 35 / 跳过 0
[1146/1334] 📥 导入

[1328/1334] 📥 导入 Sir 32 ... ✅ 新建 28 / 写入 0 / 跳过 0
[1329/1334] 📥 导入 2Jn 1 ... ✅ 新建 0 / 写入 13 / 跳过 0
[1330/1334] 📥 导入 Pro 7 ... ✅ 新建 0 / 写入 27 / 跳过 0
[1331/1334] 📥 导入 Ezk 26 ... ✅ 新建 0 / 写入 21 / 跳过 0
[1332/1334] 📥 导入 2K 1 ... ✅ 新建 0 / 写入 18 / 跳过 0
[1333/1334] 📥 导入 2S 19 ... ✅ 新建 44 / 写入 0 / 跳过 0
[1334/1334] 📥 导入 2S 7 ... ✅ 新建 0 / 写入 29 / 跳过 0

🎉 中文经文导入完成！
   ➕ 新建 verse（英文未导入）：5770
   ✍️ 写入 text_cn：29847
   ⏭️ 跳过已有中文：0
   📦 跳过已完成章节：4
